# Разработка алгоритма для сентимент анализа отзывов к медицинским учреждениям

## Сентимент анализ собранного датасета

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification
from transformers import BertTokenizerFast
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

/Users/almanelis/dev/ABSA-for-Healthcare-Reviews/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('blanchefort/rubert-base-cased-sentiment')
model = AutoModelForSequenceClassification.from_pretrained('blanchefort/rubert-base-cased-sentiment', return_dict=True)

@torch.no_grad()
def sentiment_predict(text):
    inputs = tokenizer(text, max_length=512, padding=True, truncation=True, return_tensors='pt')
    outputs = model(**inputs)
    predicted = torch.nn.functional.softmax(outputs.logits, dim=1)
    predicted = torch.argmax(predicted, dim=1).numpy()
    if predicted == 0:
        overall_sentiment = 'Neutral'
    elif predicted == 1:
        overall_sentiment = 'Positive'
    else:
        overall_sentiment = 'Negative'
    return overall_sentiment


In [ ]:
from pyabsa import AspectTermExtraction as ATEPC

aspect_extractor = ATEPC.AspectExtractor('multilingual')

In [12]:
def aspect_analysis(text):
    try:
        atepc_result = aspect_extractor.predict(text, print_result=False, save_result=False)
        aspects = atepc_result['aspect']
        sentiments = atepc_result['sentiment']
        confidences = atepc_result['confidence']

        if not aspects:
            return None

        # Возвращаем список аспектов в виде строки JSON-подобного формата
        return "; ".join([
            f"{asp}: {sent} (увер. {round(conf, 3)})"
            for asp, sent, conf in zip(aspects, sentiments, confidences)
        ])
    except Exception as e:
        return f"Ошибка анализа: {e}"


In [ ]:
df = pd.read_excel('datasets/reviews_1098_overall.xlsx')

df['общая_тональность'] = df['review_text'].apply(sentiment_predict)
df['аспектный_анализ'] = df['review_text'].apply(aspect_analysis)

df.to_excel('datasets/reviews_with_sentiment_aspects.xlsx', index=False)

/Users/almanelis/dev/ABSA-for-Healthcare-Reviews/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
